In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd

# Define NTNU study programs and their URLs
ntnu_programs = {
    "Bachelor i programmering": "https://www.ntnu.no/studier/studieplan#programmeCode=BPROG&year=2024",
    "Bachelor i informatikk": "https://www.ntnu.no/studier/studieplan#programmeCode=BIT&year=2024",
    "Bachelor i ingenioerfag, data": "https://www.ntnu.no/studier/studieplan#programmeCode=BIDATA&year=2024&dir=BIDATA-24-TRHEIM"
}

# Output rows
all_rows = []

# Set up Selenium with headless Chrome
options = Options()
options.add_argument('--headless')
driver = webdriver.Chrome(options=options)

for study_program, url in ntnu_programs.items():
    print(f"🔍 Extracting for: {study_program}")
    driver.get(url)
    time.sleep(3)

    # Force select 2024 from dropdown
    try:
        select_element = driver.find_element(By.CSS_SELECTOR, "select[aria-labelledby='velg_kull']")
        select = Select(select_element)
        select.select_by_value("2024")
        print("✅ Selected 2024")
        time.sleep(2)
    except Exception as e:
        print(f"⚠️ Failed to select year for {study_program}: {e}")

    # Extract the loaded page
    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    rows = soup.find_all("tr")

    found_any = False
    for row in rows:
        if "O" in row.text:
            cols = row.find_all("td")
            if cols:
                subject_code = cols[0].get_text(strip=True)
                subject_name = cols[1].get_text(strip=True) if len(cols) > 1 else ""
                all_rows.append({
                    "school": "NTNU",
                    "study_program": study_program,
                    "type": "Obligatorise_emner",
                    "mandatory_subject": f"{subject_code} {subject_name}"
                })
                found_any = True

    if not found_any:
        all_rows.append({
            "school": "NTNU",
            "study_program": study_program,
            "type": "Obligatorise_emner",
            "mandatory_subject": ""
        })

driver.quit()

# Save to file
df = pd.DataFrame(all_rows, columns=["school", "study_program", "type", "mandatory_subject"])
df.to_csv("Mandatory_subjects.csv", mode='a', header=False, index=False)
print("✅ Appended NTNU data to Mandatory_subjects.csv")


🔍 Extracting for: Bachelor i programmering
✅ Selected 2024
🔍 Extracting for: Bachelor i informatikk
✅ Selected 2024
🔍 Extracting for: Bachelor i ingenioerfag, data
✅ Selected 2024
✅ Appended NTNU data to Mandatory_subjects.csv
